In [1]:
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
from google import genai #type: ignore
from google.genai import types #type: ignore
from google.colab import drive #type: ignore
from dotenv import load_dotenv
from typing import Callable
import os
import asyncio
from enum import Enum

drive.mount("/content/drive")
load_dotenv("/content/drive/MyDrive/.env")
api_key = os.getenv("GOOGLE_API_KEY")

Mounted at /content/drive


In [2]:
client = genai.Client(api_key=api_key)
MODEL_ID = "gemini-2.5-flash"

In [3]:
def search(query: str) -> str:
    """Search for information about a specific topic or query.

    Args:
        query (str): The search query or topic to look up.
    """
    query_lower = query.lower()

    # Predefined responses for demonstration
    if all(word in query_lower for word in ["capital", "france"]):
        return "Paris is the capital of France and is known for the Eiffel Tower."
    elif "react" in query_lower:
        return "The ReAct (Reasoning and Acting) framework enables LLMs to solve complex tasks by interleaving thought generation, action execution, and observation processing."

    # Generic response for unhandled queries
    return f"Information about '{query}' was not found."

In [4]:
TOOL_REGISTRY: dict[str, Callable[..., str]] = {
    search.__name__: search,
}

In [5]:
def build_tools_xml_description(tool_registry: dict[str, Callable[..., str]]) -> str:
    """Build a minimal XML description of tools using only their docstrings."""
    lines = []
    for tool_name, fn in tool_registry.items():
        doc = (fn.__doc__ or "").strip()
        lines.append(f'\t<tool name="{tool_name}">')
        if doc:
            lines.append("\t\t<description>")
            for line in doc.split("\n"):
                lines.append(f"\t\t\t{line}")
            lines.append("\t\t</description>")
        lines.append("\t</tool>")
    return "\n".join(lines)


# Build a string of XML describing the tools
tools_xml = build_tools_xml_description(TOOL_REGISTRY)

PROMPT_TEMPLATE_THOUGHT = """
You are deciding the next best step for reaching the user goal. You have some tools available to you.

Available tools:
<tools>
{tools_xml}
</tools>

Conversation so far:
<conversation>
{conversation}
</conversation>

State your next **thought** about what to do next as one short paragraph focused on the next action you intend to take and why.
Avoid repeating the same strategies that didn't work previously. Prefer different approaches.

Remember:
- Return ONLY plain natural language text.
- Do NOT emit JSON, XML, function calls, or code.
""".strip()

In [6]:
print(PROMPT_TEMPLATE_THOUGHT.format(tools_xml=tools_xml, conversation=""))

You are deciding the next best step for reaching the user goal. You have some tools available to you.

Available tools:
<tools>
	<tool name="search">
		<description>
			Search for information about a specific topic or query.
			
			    Args:
			        query (str): The search query or topic to look up.
		</description>
	</tool>
</tools>

Conversation so far:
<conversation>

</conversation>

State your next **thought** about what to do next as one short paragraph focused on the next action you intend to take and why.
Avoid repeating the same strategies that didn't work previously. Prefer different approaches.

Remember:
- Return ONLY plain natural language text.
- Do NOT emit JSON, XML, function calls, or code.


In [7]:
def generate_thought(conversation: str, tool_registry: dict[str, Callable[..., str]]) -> str:
    """Generate a thought as plain text (no structured output)."""
    tools_xml: str = build_tools_xml_description(tool_registry)
    prompt: str = PROMPT_TEMPLATE_THOUGHT.format(tools_xml=tools_xml, conversation=conversation)

    response: types.GenerateContentResponse = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
    )
    return response.text.strip()

In [8]:
question = "What is the capital of France?"
print(generate_thought(question, TOOL_REGISTRY))

I need to find the capital of France to answer the user's question. The `search` tool can be used to retrieve this information.


In [12]:
PROMPT_TEMPLATE_ACTION = """
You are selecting the best next action to reach the user goal.

Conversation so far:
<conversation>
{conversation}
</conversation>

Respond either with a tool call (with arguments) or a final answer, but only if you can confidently conclude.
""".strip()

# Dedicated prompt used when we must force a final answer
PROMPT_TEMPLATE_ACTION_FORCED = """
You must now provide a final answer to the user.

Conversation so far:
<conversation>
{conversation}
</conversation>

Provide a concise final answer that best addresses the user's goal.
""".strip()

In [9]:
class ToolCallRequest(BaseModel):
    """A request to call a tool with its name and arguments."""
    tool_name: str = Field(description="The name of the tool to call.")
    arguments: dict = Field(description="The arguments to pass to the tool.")


class FinalAnswer(BaseModel):
    """A final answer to present to the user when no further action is needed."""
    text: str = Field(description="The final answer text to present to the user.")

In [10]:
def generate_action(
    conversation: str, tool_registry: dict[str, Callable[..., str]] | None = None, force_final: bool = False
) -> ToolCallRequest | FinalAnswer:
    """Generate an action by passing tools to the LLM and parsing function calls or final text.

    When force_final is True or no tools are provided, the model is instructed to produce a final answer
    and tool calls are disabled.
    """
    # Use a dedicated prompt when forcing a final answer or no tools are provided
    if force_final or not tool_registry:
        prompt: str = PROMPT_TEMPLATE_ACTION_FORCED.format(conversation=conversation)
        response = client.models.generate_content(model=MODEL_ID, contents=prompt)
        return FinalAnswer(text=response.text.strip())

    # Default action prompt
    prompt = PROMPT_TEMPLATE_ACTION.format(conversation=conversation)

    # Provide the available tools to the model; disable auto-calling so we can parse and run ourselves
    tools: list[Callable[..., str]] = list(tool_registry.values())
    config = types.GenerateContentConfig(
        tools=tools, automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
    )
    response: types.GenerateContentResponse = client.models.generate_content(
        model=MODEL_ID, contents=prompt, config=config
    )
    
		# From the reponse, we parse each "part" and check if it's a function call
    candidate = response.candidates[0]
    for part in candidate.content.parts:
        if getattr(part, "function_call", None):
            name = part.function_call.name
            args = dict(part.function_call.args or {})
            return ToolCallRequest(tool_name=name, arguments=args)

    # Otherwise, it's a final answer
    final_answer = "".join(part.text for part in candidate.content.parts)
    return FinalAnswer(text=final_answer.strip())

In [13]:
question = "What is the capital of France?"
print(generate_action(question, TOOL_REGISTRY))

tool_name='search' arguments={'query': 'capital of France'}


In [14]:
def observe(action_request: ToolCallRequest, tool_registry: dict[str, Callable[..., str]]) -> str:
    """
    Execute the selected tool and return the observation text
    (either a result or an error message)
    """
    name = action_request.tool_name
    args = action_request.arguments

    if name not in tool_registry:
        return f"Unknown tool '{name}'. Available: {', '.join(tool_registry)}"

    try:
        return tool_registry[name](**args)
    except Exception as e:
        return f"Error executing tool '{name}': {e}"

In [15]:
action_request = generate_action(conversation="What is the capital of France?", tool_registry=TOOL_REGISTRY)
print(observe(action_request, TOOL_REGISTRY))

Paris is the capital of France and is known for the Eiffel Tower.


In [ ]:
class MessageRole(str, Enum):
    """Enumeration for the different roles a message can have."""
    USER = "user"
    THOUGHT = "thought"
    TOOL_REQUEST = "tool request"
    OBSERVATION = "observation"
    FINAL_ANSWER = "final answer"
    
class Message(BaseModel):
    """A message with a role and content, used for all message types."""
    role: MessageRole = Field(
		    description="The role of the message in the ReAct loop."
		)
    content: str = Field(description="The textual content of the message.")

    def __str__(self) -> str:
        """Provides a user-friendly string representation of the message."""
        return f"{self.role.value.capitalize()}: {self.content}"

In [ ]:
class Scratchpad:
    """Container for ReAct messages with optional pretty-print on append."""

    def __init__(self, max_turns: int) -> None:
        self.messages: list[Message] = []
        self.max_turns: int = max_turns
        self.current_turn: int = 1

    def set_turn(self, turn: int) -> None:
        self.current_turn = turn

    def append(
		    self,
		    message: Message,
		    verbose: bool = False,
		    is_forced_final_answer: bool = False
		) -> None:
        self.messages.append(message)
        if verbose:
            role_to_color = {
                MessageRole.USER: pretty_print.Color.RESET,
                MessageRole.THOUGHT: pretty_print.Color.ORANGE,
                MessageRole.TOOL_REQUEST: pretty_print.Color.GREEN,
                MessageRole.OBSERVATION: pretty_print.Color.YELLOW,
                MessageRole.FINAL_ANSWER: pretty_print.Color.CYAN,
            }
            header_color = role_to_color.get(
		            message.role,
		            pretty_print.Color.YELLOW
		        )
            pretty_print_message(
                message=message,
                turn=self.current_turn,
                max_turns=self.max_turns,
                header_color=header_color,
                is_forced_final_answer=is_forced_final_answer,
            )

    def to_string(self) -> str:
        return "\n".join(str(m) for m in self.messages)


In [ ]:
def react_agent_loop(
    initial_question: str, tool_registry: dict[str, Callable[..., str]], max_turns: int = 5, verbose: bool = False
) -> str | None:
    """
    Implements the main ReAct (Thought -> Action -> Observation) control loop.
    Uses a unified message class for the scratchpad.
    """
    scratchpad = Scratchpad(max_turns=max_turns)

    # Add the user's question to the scratchpad
    user_message = Message(role=MessageRole.USER, content=initial_question)
    scratchpad.append(user_message, verbose=verbose)

    for turn in range(1, max_turns + 1):
        scratchpad.set_turn(turn)

In [ ]:
# Generate a thought based on the current scratchpad
thought_content = generate_thought(
    scratchpad.to_string(),
    tool_registry,
)
thought_message = Message(role=MessageRole.THOUGHT, content=thought_content)
scratchpad.append(thought_message, verbose=verbose)

In [ ]:
# Generate an action based on the current scratchpad
action_result = generate_action(
    scratchpad.to_string(),
    tool_registry=tool_registry,
)

In [ ]:
# If the model produced a final answer, return it
if isinstance(action_result, FinalAnswer):
    final_answer = action_result.text
    final_message = Message(
		    role=MessageRole.FINAL_ANSWER,
		    content=final_answer
		)
    scratchpad.append(final_message, verbose=verbose)
    return final_answer

In [ ]:
# Otherwise, it is a tool request
if isinstance(action_result, ToolCallRequest):
    # Log the tool request
    params_str = ", ".join(f"{k}={repr(v)}" for k, v in action_result.arguments.items())
    scratchpad.append(
        Message(role=MessageRole.TOOL_REQUEST, content=f"{action_result.tool_name}({params_str})"),
        verbose=verbose,
    )

    # Execute and capture the observation
    observation_content = observe(action_result, tool_registry)

    # Log the observation
    scratchpad.append(
        Message(role=MessageRole.OBSERVATION, content=observation_content),
        verbose=verbose,
    )

In [ ]:
# Check if the maximum number of turns has been reached. If so, force the action selector to produce a final answer
if turn == max_turns:
    forced_action = generate_action(
        scratchpad.to_string(),
        force_final=True,
    )
    if isinstance(forced_action, FinalAnswer):
        final_answer = forced_action.text
    else:
        final_answer = "Unable to produce a final answer within the allotted turns."
    final_message = Message(
		    role=MessageRole.FINAL_ANSWER,
		    content=final_answer
		)
    scratchpad.append(
		    final_message,
			   verbose=verbose,
			   is_forced_final_answer=True
		)
    return final_answer

In [ ]:
question = "What is the capital of France?"
final_answer = react_agent_loop(
		question, TOOL_REGISTRY, max_turns=2, verbose=True
)